# ЛР3. Сервер статистики SQLite

Тетрадь подготовлена на основе предыдущей ЛР и требований ЛР3.

In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import sqlite3
import datetime
import os

DB_FILE = "log.db"

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

def _connect():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""CREATE TABLE IF NOT EXISTS events (id INTEGER PRIMARY KEY AUTOINCREMENT, event_time TEXT NOT NULL, event_type TEXT NOT NULL, duration REAL)""")
    conn.commit()
    return conn

def log_event(event_type, event_time=None, duration=None):
    if not event_time:
        event_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
    conn = _connect()
    conn.execute("INSERT INTO events(event_time,event_type,duration) VALUES(?,?,?)", (str(event_time), str(event_type), None if duration in (None, '') else float(duration)))
    conn.commit(); conn.close()
    return True

def get_events(event_type='', start_time='', end_time='', min_duration='', max_duration=''):
    conn = _connect()
    query = "SELECT id,event_time,event_type,duration FROM events WHERE 1=1"; params=[]
    if event_type: query += " AND event_type=?"; params.append(event_type)
    if start_time: query += " AND event_time>=?"; params.append(start_time)
    if end_time: query += " AND event_time<=?"; params.append(end_time)
    if min_duration not in ('',None): query += " AND duration>=?"; params.append(float(min_duration))
    if max_duration not in ('',None): query += " AND duration<=?"; params.append(float(max_duration))
    query += " ORDER BY id"
    rows=conn.execute(query,params).fetchall(); conn.close()
    return [list(r) for r in rows]

def clear_log():
    conn=_connect(); conn.execute("DELETE FROM events"); conn.commit(); conn.close(); return True

def get_stats():
    conn=_connect(); r=conn.execute("SELECT COUNT(*),COALESCE(AVG(duration),0),COALESCE(MIN(duration),0),COALESCE(MAX(duration),0) FROM events WHERE duration IS NOT NULL").fetchone(); conn.close()
    return {"count":r[0],"avg_duration":r[1],"min_duration":r[2],"max_duration":r[3]}

server=SimpleXMLRPCServer(("localhost",8009),requestHandler=RequestHandler,allow_none=True)
server.register_introspection_functions()
for fn,name in [(log_event,'log_event'),(get_events,'get_events'),(clear_log,'clear_log'),(get_stats,'get_stats')]: server.register_function(fn,name)
_connect()
print("Сервер статистики: порт 8009; БД log.db")
server.serve_forever()


Сервер статистики: порт 8009; БД log.db


127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:42] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:47] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:48] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:48] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:50:53] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202